### Test AI Agent 🤖 Tool Calling with DeepEval 🧪

Testing AI Agent involves testing of the Tools being invoked by an AI Agent. Here, AI Agent will invoke the necessary tools based on the given input and respond with the help of the tools being bounded with the AI Agent


<img src="./img/AIAGent.png" width="800" height="400" style="display: block; margin: auto;">

In [5]:
#!pip install -qU duckduckgo-search

In [6]:
import deepeval

deepeval.login("confident_us_8k9P7QpyyKgjpa7yzXG0ULlki3JAwq0DPAstgNKA1x0=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [21]:
!deepeval set-ollama qwen2.5:latest

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `qwen2.5:latest` for 
all evals that require an LLM.


In [23]:
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import LLMTestCase
from deepeval.test_case import ToolCall
from deepeval.tracing import (
    observe,
    update_current_span
)

- 📦 deepeval.tracing.observe
    - A decorator that wraps a function so DeepEval can trace its execution.
    - Creates a span in the trace for that function call.
- 📦 deepeval.tracing.update_current_span
    - Lets you add attributes or events to the currently active span.
    - Replaces older fixed attribute classes (like RetrieverAttributes, which was removed)

In [24]:
from langchain_ollama import ChatOllama

@observe(type='llm', model='qwen2.5:latest') 
def local_llms(): # To use the @observe decorator, the function must take no arguments and function is needed. 
    return ChatOllama(
        base_url="http://localhost:11434",
        model = "qwen2.5:latest",
        temperature=0.5,
        max_tokens = 250
    )

llm = local_llms()

#### AI Agent with Tools

In [ ]:
from langchain.tools import tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun
from deepeval.models import OllamaModel

s_tool = DuckDuckGoSearchRun()

@tool
@observe(type='tool')
def add_numbers(a: int, b: int) -> int:
    "Add two numbers and return results."
    result = int(a) + int(b)  # Earlier we were reuturning int, but that Confident AI needs to be a String or json. 
    return f"The sum of {a} and {b} is {result}"

@tool
@observe(type='tool')
def subtract_numbers(a: int, b: int) -> int:
    "Subtract two numbers and return results."
    result = int(a) - int(b)
    return f"The difference of {a} and {b} is {result}"

@tool
@observe(type='tool')
def search_tool(query):
    "Tool to search online for the given query and return results"
    return s_tool.run(query)

tools = [add_numbers, subtract_numbers, search_tool]

# We have some predefoinied observe types like 'llm', 'tool', 'agent' etc. But if not passed, then it will be 'custom'.
@observe(type='agent', available_tools=["add_numbers", "subtract_numbers", "search_tool"], metrics=[ToolCorrectnessMetric()])
def main_ai_agent(query):
    agent = initialize_agent(
        tools= tools,
        llm=llm,
        agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
        verbose=True,
        return_intermediate_steps=True
    )
    
    response = agent.invoke(query)
    
    # The use of below update_current_span is optional, but it helps in associating the test case with the current execution span for better traceability.
    update_current_span(
        test_case=LLMTestCase(
            input=query,
            tools_called=[ToolCall(name="add_numbers")],
            expected_tools=[ToolCall(name="add_numbers")],
            actual_output=response['output']
        )
    )
    
    return response


[Confident AI Trace Log]  Successfully posted trace (0 traces remaining in 
queue, 1 in flight): 
https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/observatory/trac
es/b7e6a8af-2b9c-4b63-9f9d-fd5e097eba42 
To disable dev logging, set CONFIDENT_TRACE_VERBOSE=0 as an environment 
variable.


In [ ]:
# search_response = main_ai_agent("Who is the current president of USA in 2025, just give the name")
# add_response = main_ai_agent("What is the sum of 20 and 90")
# sub_response = main_ai_agent("What is the subtract of 100 with 50")

### Evaluating AI Agent with DeepEval for Component Testing

In [26]:
from deepeval.dataset import Golden
from deepeval import evaluate

# goldens = Golden(input="What is the sum of 20 and 90")
# evaluate(goldens=[goldens], observed_callback=main_ai_agent)


test_case = LLMTestCase(
    input="What is the sum of 20 and 90",
    expected_output="110",  # Optional, if you want to compare against a known correct answer
    actual_output="110" 
)


- evaluate() is now reserved for batch test case evaluation, not agent callbacks.
- Agent workflows are traced with observe() and evaluated using metrics like AnswerRelevancyMetric, TaskCompletionMetric, etc.
- You manually pass the agent’s output to the metric’s .measure() method.


In [30]:
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models import OllamaModel

# Run the agent
response = main_ai_agent(test_case.input)
ollama_model = OllamaModel(model="llama3.2:latest")
metric = AnswerRelevancyMetric(model=ollama_model)
# Evaluate by passing the test case directly
score = metric.measure(test_case)

print("Answer Relevancy Score:", score)



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "add_numbers",
  "action_input": {
    "a": 20,
    "b": 90
  }
}
```
Observation: The sum of 20 and 90 is 110
Thought:Action:
```
{
  "action": "Final Answer",
  "action_input": "The sum of 20 and 90 is 110"
}
```

> Finished chain.


[Confident AI Trace Log]  Successfully posted trace (0 traces remaining in queue, 1 in flight): 
https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/observatory/traces/794f65e1-ad07-4c12-8883-0c253bf93
feb 
To disable dev logging, set CONFIDENT_TRACE_VERBOSE=0 as an environment variable.

Answer Relevancy Score: 0.8


[Confident AI Metric Data Log] Successfully posted metric data (0 metrics 
remaining in queue, 1 in flight) 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an 
environment variable.
